# Sound representation probe: M00–M30

Choose **Runtime → Run all**. Upload the Phase 1 output ZIP when prompted. This notebook runs only the Sound model-representation probe; no child behavior, age, or RT is used.

In [ ]:
import pathlib, subprocess, sys, zipfile
from google.colab import files

REPOSITORY = 'https://github.com/ss-sebastian/developmental_checkpoints_word_recognition.git'
PROJECT = pathlib.Path('/content/developmental_checkpoints_word_recognition')
if not PROJECT.exists():
    subprocess.run(['git', 'clone', '--depth', '1', REPOSITORY, str(PROJECT)], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e', str(PROJECT)], check=True)
print('Project:', PROJECT)
subprocess.run(['git', '-C', str(PROJECT), 'rev-parse', 'HEAD'], check=True)

In [ ]:
import torch
assert torch.cuda.is_available(), 'Select a GPU runtime, then run all again.'
print('CUDA:', torch.cuda.get_device_name(0))
print('PyTorch:', torch.__version__)

In [ ]:
uploaded = files.upload()
zip_names = [name for name in uploaded if name.lower().endswith('.zip')]
assert len(zip_names) == 1, 'Upload exactly one Phase 1 ZIP.'
CHECKPOINT_ROOT = pathlib.Path('/content/phase1_checkpoint_bundle')
CHECKPOINT_ROOT.mkdir(parents=True, exist_ok=True)
with zipfile.ZipFile(zip_names[0]) as archive:
    archive.extractall(CHECKPOINT_ROOT)
print('Extracted bundle:', CHECKPOINT_ROOT)

In [ ]:
import os
os.chdir(PROJECT)
STIMULI = PROJECT / 'data/sound_representation_probe/primary_manifest.tsv'
OUTPUT = pathlib.Path('/content/sound_representation_probe_results')
assert STIMULI.exists(), f'Missing primary Sound manifest: {STIMULI}'
subprocess.run([
    'devlm-sound-probe', 'run',
    '--stimuli', str(STIMULI),
    '--checkpoints-dir', str(CHECKPOINT_ROOT),
    '--output-dir', str(OUTPUT),
    '--device', 'cuda',
], check=True)
print('Results:', OUTPUT)

In [ ]:
import pandas as pd
from IPython.display import display

summary = pd.read_csv(OUTPUT / 'checkpoint_summary.tsv', sep='\t')
development = pd.read_csv(OUTPUT / 'developmental_delta_auc.tsv', sep='\t')
display(summary)
display(development)

In [ ]:
import shutil
RESULTS_ZIP = shutil.make_archive('/content/sound_representation_probe_results', 'zip', OUTPUT)
files.download(RESULTS_ZIP)